In [2]:
import pandas as pd

ds1 = pd.read_csv("datasets/ds1_main.csv")

ds1.columns

Index(['student_id', 'age', 'gender', 'grade_level', 'study_hours_per_day',
       'uses_ai', 'ai_usage_time_minutes', 'ai_tools_used', 'ai_usage_purpose',
       'ai_dependency_score', 'ai_generated_content_percentage',
       'ai_prompts_per_week', 'ai_ethics_score', 'last_exam_score',
       'assignment_scores_avg', 'attendance_percentage',
       'concept_understanding_score', 'study_consistency_index',
       'improvement_rate', 'sleep_hours', 'social_media_hours',
       'tutoring_hours', 'class_participation_score', 'final_score', 'passed',
       'performance_category'],
      dtype='object')

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

ds1 = pd.read_csv("datasets/ds1_main.csv")

# ------------------------
# 1️⃣ Define baseline features
# ------------------------
features = [
    'age', 'gender', 'grade_level', 'study_hours_per_day',
    'uses_ai', 'ai_usage_time_minutes', 'ai_tools_used',
    'ai_usage_purpose', 'ai_dependency_score',
    'ai_generated_content_percentage', 'ai_prompts_per_week',
    'ai_ethics_score', 'last_exam_score',
    'assignment_scores_avg', 'attendance_percentage',
    'concept_understanding_score', 'study_consistency_index',
    'improvement_rate', 'sleep_hours', 'social_media_hours',
    'tutoring_hours', 'class_participation_score'
]

target = "final_score"

X = ds1[features]
y = ds1[target]

# ------------------------
# 2️⃣ Split numeric / categorical
# ------------------------
numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# ------------------------
# 3️⃣ Strong model
# ------------------------
model = Pipeline([
    ("preprocess", preprocess),
    ("regressor", HistGradientBoostingRegressor(
        random_state=42,
        max_depth=6,
        learning_rate=0.05,
        max_iter=600
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Baseline RMSE:", rmse)
print("Baseline R2:", r2)

Baseline RMSE: 5.2038455829820816
Baseline R2: 0.8543888434177243
